# Unified Schema Creator

Builds the per-ticker schema **`unified_schema_<ticker>`** that feeds the model.
Holds all the unified-layer logic moved out of `data_preprocessor.py`, ported to
this branch's **Simplize-primary + CafeF + TradingView** stock schema (with GICS
classification).

**Naming**: `__` separates levels, `_` inside a name.
- `pool__<group>` — pre-selection pools: `pool__basic`, `pool__calendar`, `pool__ta`, `pool__macro`, `pool__targets`
- `<target>__<group>` — post-selection (features chosen for a target)
- `<target>__final` — a VIEW joining the pieces (input of `train_test_creator.ipynb`)

**Groups**: `basic` (identity/GICS/price/microstructure) · `calendar` · `ta` (technical) · `macro` (economy/bonds/indices/sector-peers) · `targets`.

## Imports

In [1]:
import os
import re
import sys
import json

import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(".."))

from logger.logger import Logger
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from dtos.tabular_database_driver_dtos.tabular_database_driver_dtos import (
    Condition,
    SqlOperator,
    DataType,
)
from data_preprocessor.data_preprocessor import DataPreprocessor
from utils.switch_handler import SwitchHandler
from utils.constants import DATABASE_MAIN_V2, SILVER_SCHEMA, GOLD_SCHEMA

load_dotenv()

True

## Parameters

In [2]:
TICKER = "vcb"
SCHEMA = f"unified_schema_{TICKER.lower()}"
REBUILD = True  # drop & recreate each table on re-run (idempotent)

TARGET_HORIZON = 5             # trading days ahead for the labels
GAIN_THRESHOLD = 0.05          # +5% for probability_gain_5pct_5day
BENCHMARK_TICKER = "VNINDEX"   # for the cross-sectional (relative) return target

# Macro context (ported from the old UNIFIED_* constants):
ECONOMY_TICKER_PREFIX = "VN"   # restrict economy series to Vietnam (None = all)
MACRO_SOURCES = ["economy", "bonds", "indices", "stocks"]  # "stocks" = GICS-sector peers

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = int(os.getenv("POSTGRES_PORT", 5432))
PG_USER = os.getenv("POSTGRES_USER", "postgres")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")

print(f"Target schema: {SCHEMA}")

Target schema: unified_schema_vcb


## Connect & ensure schema

In [3]:
logger = Logger(file_name="../../logs/unified_schema_creator")

# DataPreprocessor is reused only for its DataFrame -> table writer
# (table creation, REAL-range sanitizing, fast COPY insert).
dp = DataPreprocessor(logger=logger, switch_handler=SwitchHandler(logger=logger))
driver = dp._database_driver
driver.connect(
    PostgreSQLConnectionDto(
        logger=logger, host=PG_HOST, user=PG_USER, password=PG_PASSWORD,
        port=PG_PORT, database=DATABASE_MAIN_V2,
    )
)
driver.create_schema(SCHEMA)


def ticker_eq(value):
    return [Condition(column="ticker", operator=SqlOperator.EQUAL_TO,
                      value=value, data_type=DataType.VARCHAR())]


print(f"Connected; schema '{SCHEMA}' ready.")

Connected; schema 'unified_schema_vcb' ready.


## Column groups, typing & save helper

In [4]:
DATE_COL = "date"

# --- basic group (matches the branch's 30-column stock schema) ---
IDENTITY = ["exchange", "ticker"]
GICS = [
    "sector", "sector_code", "industry_group", "industry_group_code",
    "industry", "industry_code", "sub_industry", "sub_industry_code",
]
BASIC_PRICE = ["open", "high", "low", "close", "net_change", "pct_change", "volume"]
BASIC_MICRO = [
    "foreign_room", "foreign_buy_volume", "foreign_sell_volume", "foreign_net_volume",
    "foreign_buy_value", "foreign_sell_value", "foreign_net_value",
    "volume_matched", "volume_negotiated", "value_matched", "value_negotiated", "foreign_own",
]
BASIC_COLS = IDENTITY + [DATE_COL] + GICS + BASIC_PRICE + BASIC_MICRO  # 30 cols

# Text columns -> VARCHAR (identity + GICS taxonomy).
TEXT_COLS = set(IDENTITY + GICS)

# Non-TA columns of gold_schema.stocks (everything else there is a TA feature).
GOLD_NON_TA = {"exchange", "ticker", DATE_COL} | set(GICS) | set(BASIC_PRICE) | set(BASIC_MICRO)

# --- calendar group ---
CALENDAR_INT = [
    "year", "quarter", "month", "week_of_year", "day_of_year", "day", "day_of_week",
    "is_month_start", "is_month_end", "is_quarter_start", "is_quarter_end",
    "is_year_start", "is_year_end",
]
CALENDAR_CYC = [
    "month_sin", "month_cos", "day_of_week_sin", "day_of_week_cos",
    "day_of_year_sin", "day_of_year_cos",
]

# Integer-typed columns -> BIGINT (volumes + integer VND foreign values + calendar).
INT_COLS = set(
    ["volume", "foreign_room", "foreign_buy_volume", "foreign_sell_volume", "foreign_net_volume",
     "volume_matched", "volume_negotiated", "foreign_buy_value", "foreign_sell_value", "foreign_net_value"]
    + CALENDAR_INT
)

# Price column per gold macro table (single-value vs OHLC convention).
GOLD_PRICE_COL = {
    "economy": "value", "bonds": "value", "forex": "value",
    "indices": "close", "funds": "close", "stocks": "close",
}


def cast_types(df, text_cols=frozenset()):
    out = df.copy()
    for c in out.columns:
        if c == DATE_COL:
            out[c] = pd.to_datetime(out[c]).dt.date
        elif c in text_cols:
            out[c] = out[c].astype("string")
        elif c in INT_COLS:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")
        else:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype(float)
    return out


def save_table(table_name, df, primary_keys=None):
    """Write a feature-group table into unified_schema_<ticker>
    (DATE / BIGINT / REAL / VARCHAR by column)."""
    primary_keys = primary_keys or [DATE_COL]
    if REBUILD:
        driver.drop_table(SCHEMA, table_name)
    overrides = {}
    for c in df.columns:
        dt = str(df[c].dtype).lower()
        if c == DATE_COL:
            overrides[c] = DataType.DATE()
        elif c in INT_COLS:
            overrides[c] = DataType.BIGINT()
        elif dt == "object" or dt.startswith("string"):
            overrides[c] = DataType.VARCHAR()
        else:
            overrides[c] = "REAL"
    dp._helper_save_pandas_table_to_database(
        schema_name=SCHEMA, table_name=table_name, primary_keys=primary_keys,
        df=df, dtype_overrides=overrides, use_copy=True,
    )
    print(f"saved {len(df)} rows x {len(df.columns)} cols -> {SCHEMA}.{table_name}")

## Unified-layer helpers (moved from `data_preprocessor.py`)

`macro_wide` is the old `_helper_macro_wide`; `sector_peers` reproduces the
GICS-sector peer resolution; `add_calendar_features` is the old
`EXTRACT_DATETIME_FEATURE`.

In [5]:
def macro_wide(table, ticker_include=None, ticker_prefix=None):
    """Pivot a gold macro table to wide form: one column per (exchange, ticker)
    series named `<table>_<exchange>_<ticker>`, indexed by date."""
    price_col = GOLD_PRICE_COL[table]
    df = driver.select(
        schema_name=GOLD_SCHEMA, table_name=table,
        columns=["exchange", "ticker", DATE_COL, price_col],
    )
    if df.empty:
        return pd.DataFrame()
    if ticker_include is not None:
        df = df[df["ticker"].isin(ticker_include)]
    if ticker_prefix is not None:
        df = df[df["ticker"].str.startswith(ticker_prefix)]
    if df.empty:
        return pd.DataFrame()
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    def _col(exchange, ticker):
        raw = f"{table}_{exchange}_{ticker}".lower()
        return re.sub(r"[^0-9a-z]+", "_", raw).strip("_")

    df["series"] = [_col(e, t) for e, t in zip(df["exchange"], df["ticker"])]
    wide = df.pivot_table(index=DATE_COL, columns="series", values=price_col, aggfunc="last")
    wide.index = pd.to_datetime(wide.index)
    return wide.sort_index()


def sector_peers(ticker):
    """Same-GICS-sector tickers (excluding self) — scope of the stocks macro join."""
    sec = driver.select(
        schema_name=GOLD_SCHEMA, table_name="stocks",
        columns=["sector"], conditions=ticker_eq(ticker), limit=1,
    )
    if sec.empty or pd.isna(sec["sector"].iloc[0]):
        return set()
    sector = sec["sector"].iloc[0]
    with driver._cursor_ctx() as cur:
        cur.execute(
            "SELECT DISTINCT ticker FROM gold_schema.stocks "
            "WHERE sector = %s AND ticker != %s",
            (sector, ticker),
        )
        peers = {r[0] for r in cur.fetchall()}
    print(f"  GICS sector='{sector}', {len(peers)} peers")
    return peers


def add_calendar_features(df, col=DATE_COL):
    dt = pd.to_datetime(df[col])
    df["year"] = dt.dt.year
    df["quarter"] = dt.dt.quarter
    df["month"] = dt.dt.month
    df["week_of_year"] = dt.dt.isocalendar().week.astype("int64")
    df["day_of_year"] = dt.dt.day_of_year
    df["day"] = dt.dt.day
    df["day_of_week"] = dt.dt.day_of_week
    df["is_month_start"] = dt.dt.is_month_start.astype(int)
    df["is_month_end"] = dt.dt.is_month_end.astype(int)
    df["is_quarter_start"] = dt.dt.is_quarter_start.astype(int)
    df["is_quarter_end"] = dt.dt.is_quarter_end.astype(int)
    df["is_year_start"] = dt.dt.is_year_start.astype(int)
    df["is_year_end"] = dt.dt.is_year_end.astype(int)
    df["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    df["day_of_week_sin"] = np.sin(2 * np.pi * dt.dt.day_of_week / 7)
    df["day_of_week_cos"] = np.cos(2 * np.pi * dt.dt.day_of_week / 7)
    df["day_of_year_sin"] = np.sin(2 * np.pi * dt.dt.day_of_year / 365)
    df["day_of_year_cos"] = np.cos(2 * np.pi * dt.dt.day_of_year / 365)
    return df

## `pool__basic`

Identity + GICS + price + microstructure, matching the branch's 30-column stock
schema. Sourced from **silver** (exact integer volumes). Keyed by `date`.

In [6]:
basic = driver.select(
    schema_name=SILVER_SCHEMA, table_name="stocks",
    columns=BASIC_COLS, conditions=ticker_eq(TICKER.upper()), order_by=[DATE_COL],
)
print(f"silver rows for {TICKER.upper()}: {len(basic)}")

basic = basic[BASIC_COLS]
basic = cast_types(basic, text_cols=TEXT_COLS)

save_table("pool__basic", basic)
basic

silver rows for VCB: 4242
saved 4242 rows x 30 cols -> unified_schema_vcb.pool__basic


,exchange,ticker,date,sector,sector_code,industry_group,industry_group_code,industry,industry_code,sub_industry,...,foreign_sell_volume,foreign_net_volume,foreign_buy_value,foreign_sell_value,foreign_net_value,volume_matched,volume_negotiated,value_matched,value_negotiated,foreign_own
0,HOSE,VCB,2009-06-30,financials,40,banks,4010,banks,401010,diversified_banks,...,0,4100,246000000,0,246000000,294070,0,17.64,0.000000e+00,NaN
1,HOSE,VCB,2009-07-01,financials,40,banks,4010,banks,401010,diversified_banks,...,3420000,170680,226055515000,215460000000,10595515000,6248390,3,389.79,2.124300e+11,NaN
2,HOSE,VCB,2009-07-02,financials,40,banks,4010,banks,401010,diversified_banks,...,104850,-49670,3280495000,6187020000,-2906525000,1515670,0,88.93,2.310000e+09,NaN
3,HOSE,VCB,2009-07-03,financials,40,banks,4010,banks,401010,diversified_banks,...,290680,-215390,4261395000,16375245000,-12113850000,899720,0,50.68,2.541000e+09,NaN
4,HOSE,VCB,2009-07-06,financials,40,banks,4010,banks,401010,diversified_banks,...,370000,-294930,4348780000,21097500000,-16748720000,1571740,0,90.18,0.000000e+00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,HOSE,VCB,2026-06-22,financials,40,banks,4010,banks,401010,diversified_banks,...,698484,-617170,4996183340,42917014582,-37920831243,2686300,0,165.05,5.940000e+09,20.22
4238,HOSE,VCB,2026-06-23,financials,40,banks,4010,banks,401010,diversified_banks,...,394290,-148955,15190921375,24414080294,-9223158919,5176000,0,320.49,0.000000e+00,20.20
4239,HOSE,VCB,2026-06-24,financials,40,banks,4010,banks,401010,diversified_banks,...,223500,-136600,5317173969,13675355375,-8358181406,2542500,0,155.57,0.000000e+00,20.20
4240,HOSE,VCB,2026-06-25,financials,40,banks,4010,banks,401010,diversified_banks,...,492439,-424339,4151385686,30019151481,-25867765795,3093500,0,188.58,0.000000e+00,20.19


## `pool__calendar`

Calendar / seasonality features derived from `date` (old
`EXTRACT_DATETIME_FEATURE`). Kept as its own group so `pool__basic` stays a clean
30-column mirror of the stock schema.

In [7]:
cal = pd.DataFrame({DATE_COL: pd.to_datetime(basic[DATE_COL])})
cal = add_calendar_features(cal)
cal = cast_types(cal)

save_table("pool__calendar", cal)
cal

saved 4242 rows x 20 cols -> unified_schema_vcb.pool__calendar


,date,year,quarter,month,week_of_year,day_of_year,day,day_of_week,is_month_start,is_month_end,is_quarter_start,is_quarter_end,is_year_start,is_year_end,month_sin,month_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos
0,2009-06-30,2009,2,6,27,181,30,1,0,1,0,1,0,0,1.224647e-16,-1.000000,0.781831,0.623490,0.025818,-0.999667
1,2009-07-01,2009,3,7,27,182,1,2,1,0,1,0,0,0,-5.000000e-01,-0.866025,0.974928,-0.222521,0.008607,-0.999963
2,2009-07-02,2009,3,7,27,183,2,3,0,0,0,0,0,0,-5.000000e-01,-0.866025,0.433884,-0.900969,-0.008607,-0.999963
3,2009-07-03,2009,3,7,27,184,3,4,0,0,0,0,0,0,-5.000000e-01,-0.866025,-0.433884,-0.900969,-0.025818,-0.999667
4,2009-07-06,2009,3,7,28,187,6,0,0,0,0,0,0,0,-5.000000e-01,-0.866025,0.000000,1.000000,-0.077386,-0.997001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,2026-06-22,2026,2,6,26,173,22,0,0,0,0,0,0,0,1.224647e-16,-1.000000,0.000000,1.000000,0.162807,-0.986658
4238,2026-06-23,2026,2,6,26,174,23,1,0,0,0,0,0,0,1.224647e-16,-1.000000,0.781831,0.623490,0.145799,-0.989314
4239,2026-06-24,2026,2,6,26,175,24,2,0,0,0,0,0,0,1.224647e-16,-1.000000,0.974928,-0.222521,0.128748,-0.991677
4240,2026-06-25,2026,2,6,26,176,25,3,0,0,0,0,0,0,1.224647e-16,-1.000000,0.433884,-0.900969,0.111659,-0.993747


## `pool__ta`

All technical-indicator features from `gold_schema.stocks` (every column there
that is not a `basic` column). Keyed by `date`.

In [8]:
ta = driver.select(
    schema_name=GOLD_SCHEMA, table_name="stocks",
    conditions=ticker_eq(TICKER.upper()), order_by=[DATE_COL],
)
ta_cols = [c for c in ta.columns if c not in GOLD_NON_TA]
ta = ta[[DATE_COL] + ta_cols]
ta = cast_types(ta)   # all TA columns -> REAL

save_table("pool__ta", ta)
print(f"ta feature columns: {len(ta_cols)}")
ta

saved 4242 rows x 906 cols -> unified_schema_vcb.pool__ta
ta feature columns: 905


,date,close_bb_20_upper,close_bb_20_middle,close_bb_20_lower,close_bb_20_dist_upper,close_bb_20_dist_middle,close_bb_20_dist_lower,close_bb_20_slope_upper,close_bb_20_slope_upper_acceleration,close_bb_20_slope_middle,...,volatility_5,volatility_21,close_roll_mean_5,close_roll_std_5,close_roll_min_5,close_roll_max_5,close_roll_mean_21,close_roll_std_21,close_roll_min_21,close_roll_max_21
0,2009-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-07-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-07-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,8919.706,271.22192,8523.951,9208.911,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,2026-06-22,63698.450,62045.0,60391.550,-0.037653,-0.012007,0.015043,-230.94698,-63.090840,-120.0,...,0.006824,0.007758,61720.000,327.10855,61300.000,62200.000,62123.810,902.16670,61300.0,64400.0
4238,2026-06-23,63170.470,61890.0,60609.530,-0.029610,-0.009533,0.011392,-527.98230,-297.035300,-155.0,...,0.006468,0.007691,61620.000,370.13510,61300.000,62200.000,62009.523,842.55850,61300.0,64400.0
4239,2026-06-24,62522.720,61730.0,60937.280,-0.024355,-0.011826,0.001029,-647.75183,-119.769550,-160.0,...,0.004665,0.007133,61380.000,277.48874,61000.000,61700.000,61847.617,669.04340,61000.0,64200.0
4240,2026-06-25,62359.656,61630.0,60900.344,-0.025011,-0.013467,-0.001648,-163.05939,484.692400,-100.0,...,0.003377,0.007134,61220.000,342.05260,60800.000,61700.000,61685.715,445.29285,60800.0,62800.0


## `pool__macro`

Macro context forward-filled onto the ticker's trading-day spine (old
`_ingest_unified_stock` join): Vietnam economy + bonds + indices + **GICS-sector
peer** closes. Causal — each day carries the last published macro value.

In [9]:
spine = driver.select(
    schema_name=GOLD_SCHEMA, table_name="stocks",
    columns=[DATE_COL], conditions=ticker_eq(TICKER.upper()), order_by=[DATE_COL],
)
spine[DATE_COL] = pd.to_datetime(spine[DATE_COL])

peers = sector_peers(TICKER.upper())

macro = spine.copy()
macro_cols = []
for table in MACRO_SOURCES:
    wide = macro_wide(
        table,
        ticker_include=peers if table == "stocks" else None,
        ticker_prefix=ECONOMY_TICKER_PREFIX if table == "economy" else None,
    )
    if wide.empty:
        print(f"  {table}: no series")
        continue
    macro = macro.merge(wide, how="left", left_on=DATE_COL, right_index=True)
    macro_cols += list(wide.columns)
    print(f"  {table}: +{len(wide.columns)} series")

macro[macro_cols] = macro[macro_cols].ffill()   # causal forward-fill
macro = cast_types(macro)   # macro series -> REAL

save_table("pool__macro", macro)
print(f"macro columns: {len(macro_cols)}")
macro

  GICS sector='financials', 37 peers
  economy: +88 series
  bonds: +18 series
  indices: +6 series
  stocks: +37 series
saved 4242 rows x 150 cols -> unified_schema_vcb.pool__macro
macro columns: 149


,date,economy_economics_vnbot,economy_economics_vnca,economy_economics_vncag,economy_economics_vncar,economy_economics_vncci,economy_economics_vncf,economy_economics_vncir,economy_economics_vncirmm,economy_economics_vncop,...,stocks_hose_vci,stocks_hose_vck,stocks_hose_vib,stocks_hose_vix,stocks_hose_vnd,stocks_hose_vpb,stocks_hose_vpx,stocks_upcom_abb,stocks_upcom_bli,stocks_upcom_hva
0,2009-06-30,-1.165000e+09,NaN,NaN,9699.0,NaN,NaN,NaN,NaN,345000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-07-01,-1.165000e+09,NaN,NaN,9699.0,NaN,NaN,NaN,NaN,345000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-07-02,-1.165000e+09,NaN,NaN,9699.0,NaN,NaN,NaN,NaN,345000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-07-03,-1.165000e+09,NaN,NaN,9699.0,NaN,NaN,NaN,NaN,345000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-07-06,-1.165000e+09,NaN,NaN,9699.0,NaN,NaN,NaN,NaN,345000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,2026-06-22,-6.700000e+08,7.654000e+09,6.4,31351.0,113.0,7.076000e+09,3.96,0.47,167000.0,...,25000.0,33300.0,16050.0,17400.0,18050.0,26000.0,27550.0,17200.0,8400.0,0.0
4238,2026-06-23,-6.700000e+08,7.654000e+09,6.4,31351.0,113.0,7.076000e+09,3.96,0.47,167000.0,...,24500.0,33400.0,16100.0,17100.0,17750.0,26550.0,27200.0,17500.0,8200.0,0.0
4239,2026-06-24,-6.700000e+08,7.654000e+09,6.4,31351.0,113.0,7.076000e+09,3.96,0.47,167000.0,...,24150.0,33200.0,16050.0,16950.0,17600.0,26550.0,27200.0,17900.0,8200.0,0.0
4240,2026-06-25,-6.700000e+08,7.654000e+09,6.4,31351.0,113.0,7.076000e+09,3.96,0.47,167000.0,...,24050.0,32950.0,16000.0,16700.0,17350.0,26700.0,27900.0,18200.0,8200.0,0.0


## `pool__targets`

The label pool (old `CREATE_TARGET`, extended to four standardized targets):

- `return_5day` — forward 5-day simple return
- `return_rel_5day` — `return_5day` minus the VNINDEX 5-day return (cross-sectional excess)
- `direction_5day` — 1 if `return_5day` > 0
- `probability_gain_5pct_5day` — 1 if close gains ≥5% within the next 5 trading days

The last `TARGET_HORIZON` rows have NaN labels (incomplete future); drop them in
`train_test_creator.ipynb`.

In [10]:
H = TARGET_HORIZON

px = driver.select(
    schema_name=GOLD_SCHEMA, table_name="stocks",
    columns=[DATE_COL, "close"], conditions=ticker_eq(TICKER.upper()), order_by=[DATE_COL],
)
px[DATE_COL] = pd.to_datetime(px[DATE_COL])
c = pd.to_numeric(px["close"], errors="coerce")

tgt = pd.DataFrame({DATE_COL: px[DATE_COL]})
tgt["return_5day"] = c.shift(-H) / c - 1.0

fwd = pd.concat([c.shift(-k) for k in range(1, H + 1)], axis=1)
fwd_max = fwd.max(axis=1)
incomplete = c.shift(-H).isna()
tgt["probability_gain_5pct_5day"] = (fwd_max / c - 1.0 >= GAIN_THRESHOLD).astype(float)
tgt.loc[incomplete, "probability_gain_5pct_5day"] = np.nan

tgt["direction_5day"] = (tgt["return_5day"] > 0).astype(float)
tgt.loc[tgt["return_5day"].isna(), "direction_5day"] = np.nan

bench = driver.select(
    schema_name=GOLD_SCHEMA, table_name="indices",
    columns=[DATE_COL, "close"], conditions=ticker_eq(BENCHMARK_TICKER), order_by=[DATE_COL],
)
if bench.empty:
    print(f"WARNING: benchmark '{BENCHMARK_TICKER}' not found; return_rel_5day = NaN.")
    tgt["return_rel_5day"] = np.nan
else:
    bench[DATE_COL] = pd.to_datetime(bench[DATE_COL])
    b = pd.to_numeric(bench["close"], errors="coerce")
    bench_ret = pd.DataFrame({DATE_COL: bench[DATE_COL], "b_ret": b.shift(-H) / b - 1.0})
    tgt = tgt.merge(bench_ret, on=DATE_COL, how="left")
    tgt["return_rel_5day"] = tgt["return_5day"] - tgt["b_ret"]
    tgt = tgt.drop(columns=["b_ret"])

tgt = tgt[[DATE_COL, "return_5day", "return_rel_5day",
           "direction_5day", "probability_gain_5pct_5day"]]
tgt = cast_types(tgt)

save_table("pool__targets", tgt)
tgt

saved 4242 rows x 5 cols -> unified_schema_vcb.pool__targets


,date,return_5day,return_rel_5day,direction_5day,probability_gain_5pct_5day
0,2009-06-30,-0.058333,-0.056861,0.0,0.0
1,2009-07-01,-0.082645,-0.121691,0.0,0.0
2,2009-07-02,-0.068966,-0.098035,0.0,0.0
3,2009-07-03,-0.080357,-0.088142,0.0,0.0
4,2009-07-06,-0.162393,-0.101636,0.0,0.0
...,...,...,...,...,...
4237,2026-06-22,NaN,NaN,NaN,NaN
4238,2026-06-23,NaN,NaN,NaN,NaN
4239,2026-06-24,NaN,NaN,NaN,NaN
4240,2026-06-25,NaN,NaN,NaN,NaN


## Next: per-target feature selection + final view

For each target (starting with `probability_gain_5pct_5day`):
1. rank `pool__ta` / `pool__macro` features against the target (XGB/SHAP + redundancy prune),
2. write survivors to `<target>__ta`, `<target>__macro` (+ a `feature_manifest` table),
3. create `<target>__final` VIEW = `pool__basic ⋈ pool__calendar ⋈ <target>__ta ⋈ <target>__macro ⋈ target col`.

That view is the input of `train_test_creator.ipynb`.

## Per-group feature selection → `<target>__lb<L>__<group>__<n>`

Rank the numeric features of a `pool__<group>` against one **target column** of
`pool__targets` (joined on `date`, target renamed to `target`), keep the
**top-`max_features`**, then **drop highly-correlated features** (|r| ≥
`corr_threshold`, default 0.9). Persist as `<target>__lb<L>__<group>__<n_kept>`
(e.g. `return_5day__lb20__calendar__6`) holding `date` + surviving features +
`target`.

- **`lookback` (`L`)** = window length for the importance. With `lookback > 1`
  each sample is a `(lookback, n_features)` window ending on day `t` with label
  `target_t` (matching a sequence model). Each window is flattened into
  `lookback × n_features` lag columns, the ensemble is fitted on those, and every
  feature's per-lag importances are **summed back into one per-feature score**.
  `lookback = 1` is the plain per-row behaviour. `L` is encoded in the table name
  (`lb<L>`) so different windows don't overwrite each other. The correlation
  prune and the saved table always use the original per-day columns, so the
  output schema is unchanged.
- **`max_features` is a cap**: the surviving count after the correlation prune is
  `≤ max_features`, and the table name carries that **real post-prune count**
  `<n_kept>`.
- **Importance** = weighted ensemble — XGBoost gain + XGBoost-SHAP + LASSO +
  ElasticNet, each min-max normalized then blended (weights in
  `FeatureSelectorType`). Regression targets (`return_5day`, `return_rel_5day`)
  and binary targets (`direction_5day`, `probability_gain_5pct_5day`) are
  **auto-detected** — tree models swap to `XGBClassifier`, linear models to
  L1/elastic-net `LogisticRegression`.
- Fitting uses rows with a **non-null target** (and, for `lookback > 1`, a full
  `lookback`-day history); the saved table keeps **all** pool rows (target NaN at
  the tail) so it still joins cleanly in the final view. Non-numeric identity /
  GICS columns are never candidates.

In [ ]:
from feature_selector.feature_selector import FeatureSelector

TARGET_COLS = ["return_5day", "return_rel_5day",
               "direction_5day", "probability_gain_5pct_5day"]


def build_selected_table(group, target, max_features, corr_threshold=0.9,
                         lookback=1, tree_only=False, device="auto", save=True):
    """Rank pool__<group> features against a pool__targets column with the
    weighted ensemble (XGB gain + SHAP + LASSO + ElasticNet, auto reg/clf),
    keep the top-`max_features`, drop |r| >= corr_threshold correlated ones, then
    persist unified_schema_<ticker>.<target>__lb<lookback>__<group>__<n_kept>.

    `lookback` = window length for the importance (lookback>1 -> per-lag
    importances summed back per feature; encoded as `lb<L>` in the name).

    `tree_only=True` skips the CPU-bound LASSO/ElasticNet CV and ranks with
    XGB gain + SHAP only (weights renormalized) — orders of magnitude faster on
    wide pools (e.g. `ta` at lookback>1). `device` ("auto"/"cuda"/"cpu") selects
    the XGBoost device; "auto" uses the GPU when XGBoost has CUDA support.

    `max_features` is a CAP on the pre-prune top-N (it does NOT change fit cost,
    which is set by n_features x lookback). The table name carries the real
    post-prune count `<n_kept>`.

    Returns the FeatureSelector SelectionResult."""
    if target not in TARGET_COLS:
        raise ValueError(f"target must be one of {TARGET_COLS}, got {target!r}")

    result = FeatureSelector(
        driver=driver, schema=SCHEMA, group=group, target=target,
        max_features=max_features, corr_threshold=corr_threshold,
        lookback=lookback, tree_only=tree_only, device=device, logger=logger,
        # Identity + GICS are constant per ticker (and the *_code columns look
        # numeric); never let them be selected as features.
        non_feature_cols=IDENTITY + GICS,
    ).run()

    n_kept = len(result.kept_features)
    result.table_name = f"{target}__lb{result.lookback}__{group}__{n_kept}"
    if save:
        frame = cast_types(result.frame, text_cols=TEXT_COLS)
        save_table(result.table_name, frame)
    return result


In [ ]:
# Example: windowed importance over a 20-day lookback for `return_5day` on
# `calendar`; keep up to 10, then correlation prune.
# -> builds unified_schema_vcb.return_5day__lb20__calendar__<n_kept>  (n_kept <= 10)
result = build_selected_table(
    group="calendar", target="return_5day", max_features=10,
    corr_threshold=0.9, lookback=20,
)
print(f"\nbuilt {SCHEMA}.{result.table_name}  (task={result.task}, lookback={result.lookback})")
print(f"kept {len(result.kept_features)}/{result.max_features}: {result.kept_features}")
print(f"dropped (corr): {result.dropped_features}")

result.importance.head(10)


## Batch: all targets × chosen groups

Build a `<target>__lb<L>__<group>__<n_kept>` table for every
`BATCH_TARGETS × BATCH_GROUPS` combination at a fixed `BATCH_LOOKBACK` /
`BATCH_MAX_FEATURES`. Failures are caught per-combination so one bad group
doesn't abort the rest; a summary DataFrame of what was built is shown at the
end.

> `ta` at `lookback > 1` is heavy (905 × L design columns → slow LASSO/ElasticNet).
> It's left out of `BATCH_GROUPS` by default — add it only if you're prepared to
> wait, or run it at `lookback=1`.

In [ ]:
# --- batch config ---
BATCH_TARGETS = TARGET_COLS                      # all 4 targets
BATCH_GROUPS = ["basic", "calendar", "macro"]    # add "ta" only with tree_only=True
BATCH_LOOKBACK = 20
BATCH_MAX_FEATURES = 100
BATCH_CORR_THRESHOLD = 0.9
BATCH_TREE_ONLY = False       # set True (or use per-group) for wide pools like `ta`
BATCH_DEVICE = "auto"         # XGBoost device: "auto" uses the GPU when available

rows = []
for target in BATCH_TARGETS:
    for group in BATCH_GROUPS:
        try:
            r = build_selected_table(
                group=group, target=target, max_features=BATCH_MAX_FEATURES,
                corr_threshold=BATCH_CORR_THRESHOLD, lookback=BATCH_LOOKBACK,
                tree_only=BATCH_TREE_ONLY, device=BATCH_DEVICE,
            )
            rows.append({
                "target": target, "group": group, "lookback": r.lookback,
                "task": r.task, "n_kept": len(r.kept_features),
                "table": r.table_name, "kept_features": ", ".join(r.kept_features),
            })
        except Exception as e:
            print(f"FAILED {target} / {group}: {e}")
            rows.append({
                "target": target, "group": group, "lookback": BATCH_LOOKBACK,
                "task": None, "n_kept": 0, "table": None, "kept_features": f"ERROR: {e}",
            })

batch_summary = pd.DataFrame(rows)
print(f"\nbuilt {(batch_summary['table'].notna()).sum()}/{len(batch_summary)} tables")
batch_summary


## `<target>__lb<L>__final` VIEW

Join every `<target>__lb<L>__<group>__<n>` selection table on `date` into one wide
VIEW — the model input. Keeps `date` once, every selected feature from all groups,
and `target` once. Built as a VIEW (not a table), per the design, so the wide join
isn't materialized and stays under PostgreSQL's 1600-column table limit.

In [ ]:
def build_final_view(target, lookback):
    """Create VIEW <target>__lb<L>__final joining all
    <target>__lb<L>__<group>__<n> selection tables on `date`
    (date once, every feature, target once). Returns the view name."""
    view = f"{target}__lb{lookback}__final"
    prefix = f"{target}__lb{lookback}__"
    with driver._cursor_ctx() as cur:
        # Exact prefix match in Python: SQL LIKE treats '_' as a single-char
        # wildcard, so a 'lb2__%' pattern would also match the 'lb20' tables
        # (duplicate columns). List all base tables and filter by startswith.
        cur.execute(
            """SELECT table_name FROM information_schema.tables
               WHERE table_schema=%s AND table_type='BASE TABLE'
               ORDER BY table_name;""",
            (SCHEMA,),
        )
        tables = [r[0] for r in cur.fetchall()
                  if r[0].startswith(prefix) and not r[0].endswith("__final")]
        if not tables:
            raise ValueError(f"no source tables matching {prefix}*")

        def feats(t):
            cur.execute(
                """SELECT column_name FROM information_schema.columns
                   WHERE table_schema=%s AND table_name=%s
                   ORDER BY ordinal_position;""",
                (SCHEMA, t),
            )
            return [c for (c,) in cur.fetchall() if c not in (DATE_COL, "target")]

        aliases = [f"a{i}" for i in range(len(tables))]
        sel = [f'{aliases[0]}."{DATE_COL}"']
        for a, t in zip(aliases, tables):
            sel += [f'{a}."{c}"' for c in feats(t)]
        sel.append(f'{aliases[0]}."target"')
        joins = [
            f'JOIN {SCHEMA}."{tables[i]}" {aliases[i]} '
            f'ON {aliases[i]}."{DATE_COL}"={aliases[0]}."{DATE_COL}"'
            for i in range(1, len(tables))
        ]
        sql = (
            f'CREATE VIEW {SCHEMA}."{view}" AS
SELECT ' + ",
  ".join(sel)
            + f'
FROM {SCHEMA}."{tables[0]}" {aliases[0]}
' + "
".join(joins) + ";"
        )
    driver.execute_query(f'DROP VIEW IF EXISTS {SCHEMA}."{view}";')
    driver.execute_query(sql)
    print(f"created VIEW {SCHEMA}.{view} from {len(tables)} tables: {tables}")
    return view


final_view = build_final_view("return_5day", 20)
final_df = driver.select(schema_name=SCHEMA, table_name=final_view, order_by=[DATE_COL])
print(f"{final_view}: {final_df.shape[0]} rows x {final_df.shape[1]} cols")
final_df